In [ ]:
import os
device = "cuda"
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('../..')

from DIGNN.data import AtomsData, ase2AtomsData
from DIGNN.utils import AtomIndexMapper
from DIGNN.pl import DataModule, TrainModule_FF, TrainModule
from DIGNN.nn import models as dgm
from DIGNN.nn.utils import init_weights


import time
import torch
import numpy as np
import pytorch_lightning as pl
from ase.build import molecule

In [ ]:
pml_rcut = 2.0
pml_mnn = 12
iml_rcut = 4.0
iml_mnn = 16

In [ ]:
atoms = molecule('H2O2')
atoms.arrays['energy'] = np.array([0.0])
atoms.arrays['force'] = np.zeros_like(atoms.positions)
atoms.arrays['formation'] = np.array([1.0])
atomsdata = [ase2AtomsData(atoms, check_rcut=pml_rcut, properties=['energy', 'force', 'formation']) for _ in range(100)]


## 力场训练测试

In [ ]:
data = DataModule(atomsdata, 
                    pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                    iml_rcut=iml_rcut, iml_mnn=iml_mnn,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=0, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='basic',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup


In [ ]:
def create_model():
    feature_dim = {'atom': 64, 'bond': 64, 'angle': 32, 'dihedral': 16}
    model = dgm.dignn.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                                atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml_rcut=pml_rcut+0.2,
                                                bondI_dim=feature_dim['bond'],
                                                iml_rcut=iml_rcut+0.2),
                    processor=dgm.GCN_Processor(atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml=1,
                                                iml=4,
                                                residual=True,
                                                dropout=0.0,
                                                bondI_dim=feature_dim['bond'],
                                                init_nn_layer=0,
                                                ), 
                    decoder=dgm.Decoder(dim=[feature_dim['atom'],64,1], 
                                        reduce_method='sum', 
                                        dropout=0.0),
                    ).to(device)
    model.apply(init_weights)
    return model

model = create_model()

In [ ]:
max_epoch = 10

train_module = TrainModule_FF(model, onecycle_total_steps=max_epoch*len(data.train_dataloader()))
trainer = pl.Trainer(max_epochs=10,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=1,
                    benchmark=True,
                    inference_mode=False,
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)


In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

## 性质训练测试

In [ ]:
data = DataModule(atomsdata, 
                    pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                    iml_rcut=iml_rcut, iml_mnn=iml_mnn,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=0, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='cplt',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

In [ ]:
def create_model():
    feature_dim = {'atom': 64, 'bond': 64, 'angle': 32, 'dihedral': 16}
    model = dgm.dignn.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                                atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml_rcut=pml_rcut+0.2,
                                                bondI_dim=feature_dim['bond'],
                                                iml_rcut=iml_rcut+0.2),
                    processor=dgm.GCN_Processor(atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml=1,
                                                iml=4,
                                                residual=True,
                                                dropout=0.0,
                                                bondI_dim=feature_dim['bond'],
                                                init_nn_layer=0,
                                                ), 
                    decoder=dgm.Decoder(dim=[feature_dim['atom'],64,1], 
                                        reduce_method='mean', 
                                        dropout=0.0),
                    ).to(device)
    model.apply(init_weights)
    return model

model = create_model()

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

class TrainModule(pl.LightningModule):
    def __init__(self,
                model,
                lr: float = 1e-3,
                prop: str = 'prop',
                adamw_weight_decay: float = 1e-2,
                adamw_betas: tuple = (0.9, 0.999),
                onecycle_total_steps: int = None,
                onecycle_final_div_factor: float = 1e+5,
                ):
        super().__init__()
        self.model = model
        self.lr = lr
        self.adamw_weight_decay = adamw_weight_decay
        self.adamw_betas = adamw_betas
        self.prop = prop

        self.onecycle_total_steps = onecycle_total_steps
        self.onecycle_final_div_factor = onecycle_final_div_factor
        
        self.criterion = torch.nn.MSELoss()
        self.mae_criterion = torch.nn.L1Loss()
        self.save_hyperparameters(ignore=["model"])

    def forward(self, cplt):
        return self.model(cplt)

    def training_step(self, batch, batch_idx):
        cplt = batch.to(self.device)
        prop = self(cplt)
        loss = self.criterion(prop.view(-1, 1), cplt[self.prop].view(-1, 1))
        
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        cplt = batch.to(self.device)
        prop = self(cplt)
        loss = self.criterion(prop.view(-1, 1), cplt[self.prop].view(-1, 1))
        mae_prop = self.mae_criterion(prop.view(-1, 1), cplt[self.prop].view(-1, 1))
        
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_mae_prop", mae_prop, prog_bar=True, on_epoch=True)
        
        return {"val_loss": loss, "val_mae_prop": mae_prop}

    def test_step(self, batch, batch_idx):
        cplt = batch.to(self.device)
        prop = self(cplt)
        loss = self.criterion(prop.view(-1, 1), cplt[self.prop].view(-1, 1))
        mae_prop = self.mae_criterion(prop.view(-1, 1), cplt[self.prop].view(-1, 1))
        
        self.log("test_loss", loss, prog_bar=True, on_epoch=True)
        self.log("test_mae_prop", mae_prop, prog_bar=True, on_epoch=True)
        
        return {"test_loss": loss, "test_mae_prop": mae_prop}
    
    def configure_optimizers(self):
        optimizer = AdamW(self.model.parameters(),
                        lr=self.lr,
                        weight_decay=self.adamw_weight_decay,
                        betas=self.adamw_betas
                        )
        
        scheduler = OneCycleLR(optimizer,
                                max_lr=self.lr,
                                total_steps=self.onecycle_total_steps,
                                final_div_factor=self.onecycle_final_div_factor,
                            )
        
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "interval": "step",},
                }

In [ ]:
max_epoch = 10

train_module = TrainModule(model, onecycle_total_steps=max_epoch*len(data.train_dataloader()), prop='formation')
trainer = pl.Trainer(max_epochs=10,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=1,
                    precision='16-mixed',
                    benchmark=True,
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())